# Rotten Tomatoes Cleaning EDA
Scott Person

## Pick Research Question
- Option A — Critics vs Audience: When do critics and audiences agree, and when do they disagree the most?
- Option B — Polarization by Category: Which genres or ratings produce the most “polarizing” movies (biggest gaps between audience and Tomatometer ratings)?
- Option C — Streaming Window Story: Has the time between theater release and streaming release changed over time, and is it related to audience or critic ratings?
- Option D — Studio Fingerprints: Do some studios systematically perform better with critics vs audiences, after controlling for sample size?

I'm going to select Option C. Have always been curious about this.

In [0]:
data_path = './data/'

In [0]:
import pandas as pd

In [0]:
df = pd.read_csv(data_path + 'Rotten Tomatoes Movies.csv')

## Cleaning Steps

### Setting Data Types

In [0]:
df.info()

In [0]:
df.head()

#### Dates to DateTime
We'll probably want to use date functions on these values, specifically days between.

In [0]:
df['in_theaters_date'] = pd.to_datetime(df['in_theaters_date'])
df['on_streaming_date'] = pd.to_datetime(df['on_streaming_date'])

print(df['in_theaters_date'].dtype)
print(df['on_streaming_date'].dtype)

####Set Categories
Categories are more memory efficient (not particularly an issue here but it's good practice), they are slightly faster, and also allow us to set up ordering that is not alphabetic.

Note: this is the wrong place to do this. If we want to fix the data we're going to be tripping over the category type. I'll try to remember to redo this later but at least want a note.

df['genre'] = df['genre'].astype('category')
df['rating'] = df['rating'].astype('category')
df['studio_name'] = df['studio_name'].astype('category')
df['tomatometer_status'] = df['tomatometer_status'].astype('category')

print(df['genre'].dtype)
print(df['rating'].dtype)
print(df['studio_name'].dtype)
print(df['tomatometer_status'].dtype)

In [0]:
# we may want to clean this up later
df['rating'].unique()

###Handling Missing Data

In [0]:
df.info()

Because of the nature of my question, I'm going to ignore nulls in many of the object fields. They don't have analytical value in my case. These fields are:
- movie_info
- critics_consensus
- genre
- directors
- writers
- cast
- studio_name

Until I have a use for them, I'm also going to ignore nulls in the following column(s):
- runtime_in_minutes
- audience_count

#### Dates

In [0]:
# Count of rows with null in_theaters_date, broken down by streaming year
null_theaters = df[df['in_theaters_date'].isna()]

null_theaters.groupby(null_theaters['on_streaming_date'].dt.year).size()

For the sake of my analysis I'm going to set all in_theaters_date values to the on_streaming_date. In the calculation this will result in a zero date difference and thus indicate direct to streaming with no theater release. Reasonable I think.

In [0]:
null_theaters = df['in_theaters_date'].isna()
df.loc[null_theaters, 'in_theaters_date'] = df.loc[null_theaters, 'on_streaming_date']

In [0]:
df[df['on_streaming_date'].isna()]

These two don't have either date nor are they widely viewed. I'm going to just drop them.

In [0]:
df = df.dropna(subset=['on_streaming_date'])
print(df['on_streaming_date'].isna().sum())

#### audience_rating

In [0]:
df[df['audience_rating'].isna()]

audience_rating is the one column that I don't know how to handle with regard to nulls. I'm going to hold of on doing anything until I get further in the EDA.

### Inconsistent Text & Tyos

In [0]:
df['genre'].unique()

I didn't see anything obvious in genre (I used other tooling too) but decided to turn it into a list of strings for each movie. This would make analysis easier in the future although not useful for my problem.

In [0]:
df['genre'] = df['genre'].str.split(', ')

In [0]:
df['rating'].unique()

Rating has a couple of values with trailing parentheis. I'm going to strip those.

In [0]:
df['rating'] = df['rating'].str.rstrip(')')
df['rating'].unique()

In [0]:
# check studio_name value counts
studio_counts = df['studio_name'].value_counts(dropna=False)
print(studio_counts)

# also check for leading/trailing whitespace
studio_stripped = df['studio_name'].str.strip()
whitespace_issues = df[df['studio_name'] != studio_stripped][['studio_name']]
print('\nRows with leading/trailing whitespace:')
print(whitespace_issues.drop_duplicates())

Roadside Attractions has one entry with a trailing space. Fix that.

In [0]:
df.loc[df['studio_name'].str.strip() != df['studio_name'], 'studio_name'] = df['studio_name'].str.strip()
print(df['studio_name'].value_counts(dropna=False))

This is a long list and some of the issues that I found anecdotally are rather complicated to see - stray periods, brackets etc. I'm going to pick a couple but cleaning this list thoroughly would be a project in itself.

In [0]:
print('weird characters')
broken = studio_counts.index.str.contains(r'\[[^\]]*\]|\*|^\(|\([^)]*$', regex=True).fillna(False)
print(studio_counts[broken].to_string())

In [0]:
df.loc[df['studio_name'] == 'Film Ventures International (F', 'studio_name'] = 'Film Ventures International'

df.loc[df['studio_name'] == 'Walt Disney Pictures [us]', 'studio_name'] = 'Walt Disney Pictures'

df.loc[df['studio_name'] == 'First Run Features (', 'studio_name'] = 'First Run Features'

df.loc[df['studio_name'] == 'IFC Center*', 'studio_name'] = 'IFC Center'

# PMK*BNC is the actual name here

df.loc[df['studio_name'] == 'Varma Corporation Limited [in]', 'studio_name'] = 'Varma Corporation Limited'

Look for unusually short studio_names values

In [0]:
print(studio_counts[studio_counts.index.str.len() <= 2].to_string())

In [0]:
short_names = studio_counts.index[studio_counts.index.str.len() <= 2]
df[df['studio_name'].isin(short_names)]

This required some research and I feel that I'm down a rabbit hole. I'll just fix one to show that I can.

WB on Mystic River is certainly Warner Bros. Fix that.

In [0]:
warner_studios = df[df['studio_name'].str.startswith('Warner', na=False)]
print(warner_studios['studio_name'].value_counts(dropna=False))
print(f'\nTotal: {len(warner_studios)}')

Yikes! That is a royal mess. Let's do a little fixing while we're doing this.

In [0]:
warner_variants = [
    'Warner Bros.',
    'Warner Bros. Animation',
    'Warner Bros',
    'Warner Brothers',
    'Warner Bros Pictures',
    'Warners Bros. Pictures',
    'WB'
]

df.loc[df['studio_name'].isin(warner_variants), 'studio_name'] = 'Warner Bros. Pictures'
print(df[df['studio_name'] == 'Warner Bros. Pictures']['studio_name'].value_counts())

OK - Hopefully I've proven that I can. Cleaning this up needs AI!!!

## Explorator Data Analysis (EDA)

In [0]:
df['days_to_streaming'] = (df['on_streaming_date'] - df['in_theaters_date']).dt.days
print(df[['in_theaters_date', 'on_streaming_date', 'days_to_streaming']].head(10))

### Plots and Outlier Elimination

In [0]:
!pip install seaborn
import seaborn as sns
import matplotlib.pyplot as plt

In [0]:
plt.figure(figsize=(10, 6))
sns.histplot(df['days_to_streaming'].dropna(), bins=30)
plt.title('Distribution of Days to Streaming')
plt.xlabel('Days to Streaming')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(8, 6))
sns.boxplot(x=df['days_to_streaming'].dropna())
plt.title('Box Plot of Days to Streaming')
plt.xlabel('Days to Streaming')
plt.tight_layout()
plt.show()

Tons of right skew! With lots of outliers out past 10,000 days. This is undoubtedly a function of date. 1935 movies didn't get streaming until streaming existed. These will throw off our numbers. So... off to research when streaming became enough of a thing to use in a valide calculation.

OK... back. I'm going to go with 2010 for the date based on readings from the following sources. Specifically that in 2010 32% of folks had watched a movie online representing a doubling since 2007. That seems like a reasonable critical mass.

Sources:
https://www.pewresearch.org/short-reads/2017/09/13/about-6-in-10-young-adults-in-u-s-primarily-use-online-streaming-to-watch-tv — Pew Research Center, About 6 in 10 young adults in U.S. primarily use online streaming to watch TV, 13 September 2017 

https://en.wikipedia.org/wiki/Timeline_of_Netflix — Timeline of Netflix, Wikipedia

https://www.pewresearch.org/internet/2010/06/03/the-state-of-online-video — Pew Research Center, The State of Online Video, 3 June 2010

 https://www.nielsen.com/insights/2010/busting-the-cord-cutting-myth-video-in-the-interactive-age — Nielsen, Busting the Cord-Cutting Myth, June 2010

In [0]:
df = df[df['in_theaters_date'] >= '2010-01-01'].copy()
print(f'Rows after excluding pre-2010 releases: {len(df)}')

In [0]:
plt.figure(figsize=(10, 6))
sns.histplot(df['days_to_streaming'].dropna(), bins=30)
plt.title('Distribution of Days to Streaming')
plt.xlabel('Days to Streaming')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(8, 6))
sns.boxplot(x=df['days_to_streaming'].dropna())
plt.title('Box Plot of Days to Streaming')
plt.xlabel('Days to Streaming')
plt.tight_layout()
plt.show()

I'm going to drop all records with a negative days_to_streaming.

In [0]:
df = df[df['days_to_streaming'] >= 0].copy()
print(f'Rows after dropping negative days_to_streaming: {len(df)}')

In [0]:
plt.figure(figsize=(10, 6))
sns.histplot(df['days_to_streaming'].dropna(), bins=30)
plt.title('Distribution of Days to Streaming')
plt.xlabel('Days to Streaming')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [0]:
plt.figure(figsize=(8, 6))
sns.boxplot(x=df['days_to_streaming'].dropna())
plt.title('Box Plot of Days to Streaming')
plt.xlabel('Days to Streaming')
plt.tight_layout()
plt.show()

In [0]:
df['release_year'] = df['in_theaters_date'].dt.year
avg_by_year = df.groupby('release_year')['days_to_streaming'].mean().sort_index()

plt.figure(figsize=(10, 6))
plt.plot(avg_by_year.index, avg_by_year.values, marker='o')
plt.title('Average Days to Streaming by Release Year')
plt.xlabel('Release Year')
plt.ylabel('Avg Days to Streaming')
plt.tight_layout()
plt.show()

In [0]:
corr_cols = ['days_to_streaming', 'release_year', 'tomatometer_rating', 'audience_rating']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

The correlation with the ratings is pretty weak. I'm going to focus on release_year for my testing.

In [0]:
# Spearman rank correlation: release_year vs days_to_streaming
# Research question: among films with a real theatrical-to-streaming window,
# does a later theatrical year come with a shorter wait?

import pandas as pd
from scipy.stats import spearmanr

work = df.copy()

# Run Spearman rank correlation
rho, p_value = spearmanr(df['release_year'], df['days_to_streaming'])
n = len(df)

print(f'n = {n}')
print(f'Spearman rho = {rho:.4f}')
print(f'p-value (two-sided) = {p_value}')
print('Note: p-value is not really 0. Let us call it < 0.001.')

# Three short sentences for the writeup
p_str = 'p < 0.001' if p_value < 0.001 else f'p = {p_value:.3f}'
print(f'1. The test is a Spearman rank correlation. The null hypothesis is that there is no monotonic association between theatrical year and days to streaming.')
print(f'2. Spearman rho = {rho:.2f}, {p_str}. The association is statistically significant.')
print(f"3. The negative rho indicates that later films tend to reach streaming sooner. The association is moderate, not strong, with rho ≈ {abs(rho):.2f}; year does not tightly predict an individual film's wait.")


## Findings and Conclusion
